## SCORE: 1.01016

## 1. Установка необходимых библиотек

In [ ]:
!pip install --upgrade featuretools >> None
!pip install --upgrade lightgbm >> None
!pip install --upgrade catboost >> None

## 2. Импорт библиотек и настройка окружения

In [ ]:
import pandas as pd
import numpy as np

from sklearn.ensemble import RandomForestRegressor

from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

from catboost import CatBoostRegressor, Pool
import lightgbm as lgb

import warnings
warnings.filterwarnings("ignore")

## 3. Загрузка данных

In [ ]:
transaction_file = '/kaggle/input/dataset-generated/dataset_generated_with_cats.csv'
dataset_generated_with_cats = pd.read_csv(transaction_file)

In [ ]:
target_file = '/kaggle/input/alfa-challenge/train.pa'
df_target = pd.read_parquet(target_file)

In [ ]:
print("Данные успешно загружены.")

## 4. Предварительная обработка данных

In [ ]:
target = df_target[['client_num', 'target']]

data_for_model = dataset_generated_with_cats.merge(target, on='client_num', how='inner')
print("Признаки и целевая переменная объединены для модели.")

cat_features = data_for_model.select_dtypes(include=['object', 'category']).columns.tolist()

X = data_for_model.drop(['client_num', 'target'], axis=1)
y = data_for_model['target']

## 5. Формирование сбалансированных выборок

In [ ]:
# Определяем уникальные классы
unique_classes = np.sort(y.unique())

# Считаем количество экземпляров каждого класса
class_counts = y.value_counts()
smallest_class_count = class_counts.min()

# Вычисляем 20% от самого маленького класса
N = int(0.2 * smallest_class_count)
print(f"Самый маленький класс имеет {smallest_class_count} экземпляров.")
print(f"Будем выбирать по {N} экземпляров каждого класса для валидационной выборки.")

valid_indices = []

# Для каждого класса выбираем N экземпляров
for c in class_counts.index:
    class_indices = y[y == c].index
    selected_indices = np.random.choice(class_indices, size=N, replace=False)
    valid_indices.extend(selected_indices)

# Получаем индексы обучающей выборки
valid_indices = np.array(valid_indices)
train_indices = y.index.difference(valid_indices)

# Формируем обучающие и валидационные выборки
X_train = X.loc[train_indices]
y_train = y.loc[train_indices]
X_valid = X.loc[valid_indices]
y_valid = y.loc[valid_indices]
print("Данные разделены на обучающую и валидационную выборки")

## 6. Расчёт весов классов

In [ ]:
class_counts_train = y_train.value_counts()
total_samples = len(y_train)
class_weights = {c: total_samples / (len(unique_classes) * class_counts_train[c]) for c in unique_classes}
print("Веса классов рассчитаны на основе обучающей выборки.")

def map_class_weights(y_labels, class_weights):
    return y_labels.map(class_weights).values

weights_train = map_class_weights(y_train, class_weights)
weights_valid = map_class_weights(y_valid, class_weights)
print("Веса для обучающей и валидационной выборок рассчитаны.")

## 7. Определение кастомной метрики WMAE

In [ ]:
class WMAEMetric:
    def get_final_error(self, error, weight):
        return error / weight

    def is_max_optimal(self):
        return False

    def evaluate(self, approxes, target, weight):
        approx = approxes[0]
        target = np.array(target)
        weight = np.ones_like(target) if weight is None else np.array(weight)
        error = np.sum(weight * np.abs(target - approx))
        return error, np.sum(weight)
        
print("Кастомная метрика WMAE определена.")

## 8. Предобработка данных для моделей

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), cat_features)
    ],
    remainder='passthrough'
)

results = {}

num_features = [col for col in X.columns if col not in cat_features]

## 9. Обучение модели LightGBM

In [ ]:
print("\nОбучение модели LightGBM...")

for col in cat_features:
    X_train[col] = X_train[col].astype('category')
    X_valid[col] = X_valid[col].astype('category')

categorical_feature_indices = [X_train.columns.get_loc(col) for col in cat_features]

lgb_train = lgb.Dataset(
    data=X_train,
    label=y_train,
    weight=weights_train,
    categorical_feature=categorical_feature_indices
)

lgb_valid = lgb.Dataset(
    data=X_valid,
    label=y_valid,
    weight=weights_valid,
    reference=lgb_train,
    categorical_feature=categorical_feature_indices
)

def wmae_metric(y_pred, dataset):
    y_true = dataset.get_label()
    weights = dataset.get_weight()
    if weights is None:
        weights = np.ones_like(y_true)
    error = np.sum(weights * np.abs(y_true - y_pred)) / np.sum(weights)
    return 'WMAE', error, False


# ПАРАМЕТРЫ БЕЗ OPTUNA
params = {
    'objective': 'regression',
    'metric': 'None',
    'depth': 11,
    'random_state': 42,
    'verbose': 10                   
}

num_round = 10_000                  
early_stopping_rounds = 100

callbacks = [
    lgb.early_stopping(stopping_rounds=early_stopping_rounds, verbose=True),
    lgb.log_evaluation(period=50)
]

bst = lgb.train(
    params=params,
    train_set=lgb_train,
    valid_sets=[lgb_valid],
    num_boost_round=num_round,
    feval=wmae_metric,
    callbacks=callbacks
)

y_valid_pred = bst.predict(X_valid, num_iteration=bst.best_iteration)

wmae = np.sum(weights_valid * np.abs(y_valid - y_valid_pred)) / np.sum(weights_valid)
print('LightGBM Validation WMAE:', wmae)

results['LightGBM'] = wmae

y_pred_bst = y_valid_pred

all_client_nums = dataset_generated_with_cats['client_num']
train_client_nums = df_target['client_num']
test_client_nums = all_client_nums[~all_client_nums.isin(train_client_nums)]

test_features = dataset_generated_with_cats[dataset_generated_with_cats['client_num'].isin(test_client_nums)]

X_test = test_features.drop(['client_num'], axis=1)

for col in cat_features:
    X_test[col] = X_test[col].astype('category')

test_pred_bst = bst.predict(X_test, num_iteration=bst.best_iteration)

## 10. Обучение модели RandomForestRegressor

In [ ]:
print("\nОбучение модели RandomForestRegressor...")

# НЕ ОБРАБОТАНО В OPTUNA
rf_model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', RandomForestRegressor(
        n_estimators=500,
        max_depth=11,
        random_state=42,
        oob_score=True,       
        bootstrap=True, 
        n_jobs=-1
    ))
])

rf_model.fit(
    X_train, y_train,
    regressor__sample_weight=weights_train
)

y_valid_pred = rf_model.predict(X_valid)

wmae = np.sum(weights_valid * np.abs(y_valid - y_valid_pred)) / np.sum(weights_valid)
print('RandomForestRegressor Validation WMAE:', wmae)

results['RandomForestRegressor'] = wmae

y_pred_rf = y_valid_pred

test_pred_rf = rf_model.predict(X_test)

## 11. Обучение модели CatBoost

In [ ]:
print("\nОбучение модели CatBoostRegressor...")

for col in cat_features:
    X_train[col] = X_train[col].astype(str)
    X_valid[col] = X_valid[col].astype(str)

train_pool = Pool(
    data=X_train, 
    label=y_train, 
    weight=weights_train, 
    cat_features=cat_features
)
valid_pool = Pool(
    data=X_valid, 
    label=y_valid, 
    weight=weights_valid, 
    cat_features=cat_features
)
print("Данные подготовлены для CatBoost.")

params = {
    'iterations': 10_000,
    'depth': 11,
    'l2_leaf_reg': 2,
    'colsample_bylevel': 0.4,
    'boosting_type': 'Plain',
    'bootstrap_type': 'MVS',
    'eval_metric': WMAEMetric(),
    'loss_function': 'MAE',
    'random_seed': 42,
    'verbose': True,
    'early_stopping_rounds': 100,
    'use_best_model': True
}
print("Параметры модели настроены.")

cat = CatBoostRegressor(**params)

cat.fit(
    train_pool,
    eval_set=valid_pool,
    verbose=True
)
print("Модель обучена.")

y_valid_pred = cat.predict(X_valid)

wmae = np.sum(weights_valid * np.abs(y_valid - y_valid_pred)) / np.sum(weights_valid)
print('CatBoost Validation WMAE:', wmae)

results['CatBoost'] = wmae

y_pred_cat = y_valid_pred

X_test[cat_features] = X_test[cat_features].astype(str)
test_pred_catboost = cat.predict(X_test)

## 12. Результаты на train для каждой модели

In [ ]:
results_df = pd.DataFrame({
    'Название модели': list(results.keys()),
    'WMAE metric': list(results.values())
})

print("\nРезультаты:")
print(results_df)

## 13. Blending

In [ ]:
best_wmae = float('inf')
best_weights = (0, 0, 0)

weights_range = np.arange(0, 1.01, 0.01)

for a in weights_range:
    for b in weights_range:
        c = 1 - a - b  
        if 0 <= c <= 1:
            y_valid_pred = (
                a * y_pred_bst +
                b * y_pred_rf +
                c * y_pred_cat
            )
            y_valid_pred_rounded = np.floor(y_valid_pred)
            wmae = np.sum(weights_valid * np.abs(y_valid - y_valid_pred_rounded)) / np.sum(weights_valid)
            if wmae < best_wmae:
                best_wmae = wmae
                best_weights = (a, b, c)

print(f"Наименьшее значение WMAE: {best_wmae}")
print(f"Наилучшие веса моделей:")
print(f"    Вес LightGBM (a): {best_weights[0]:.2f}")
print(f"    Вес Random Forest (b): {best_weights[1]:.2f}")
print(f"    Вес CatBoost (c): {best_weights[2]:.2f}")

## 14. Формирование submission

In [ ]:
test_pred_blended = (
    a * test_pred_bst +
    b * test_pred_rf +
    c * test_pred_catboost
)

test_predictions_rounded = np.round(test_pred_blended).astype(int)

submission = pd.DataFrame({
    'client_num': test_features['client_num'].values,
    'target': test_predictions_rounded
})

submission.to_csv('test_blended_predictions.csv', index=False)
print("Предсказания сохранены в файл 'test_blended_predictions.csv'.")